# UP-Fall: Kaggle data preparation and visual inspection

This first notebook stage only performs:

1. ZIP discovery
2. extraction into Kaggle's writable working directory
3. subject/activity/trial/camera parsing
4. chronological PNG sorting
5. metadata construction
6. RGB sequence inspection

RTMPose is deliberately left for the next verified stage.

In [ ]:
from pathlib import Path
import re
import shutil
import zipfile

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

pd.set_option('display.max_colwidth', 120)
print('Imports complete.')

## 1. Configuration

After attaching the UP-Fall dataset to the Kaggle notebook, set `DATASET_ROOT` to its folder under `/kaggle/input`. Leave `ZIP_SEARCH_ROOT` unchanged unless the ZIPs are inside a more specific subfolder.

In [ ]:
# TODO: fill this in after attaching the Kaggle dataset.
# Example: DATASET_ROOT = Path('/kaggle/input/up-fall-detection-dataset')
DATASET_ROOT = Path('')

ZIP_SEARCH_ROOT = DATASET_ROOT
EXTRACT_ROOT = Path('/kaggle/working/up_fall_extracted')
METADATA_CSV = Path('/kaggle/working/up_fall_metadata.csv')

# Current development subset. Set to None later to retain every available value.
KEEP_SUBJECTS = {1, 2, 3}
KEEP_ACTIVITIES = set(range(1, 12))
KEEP_TRIALS = {1}
KEEP_CAMERAS = {1}

if str(DATASET_ROOT) in {'', '.'}:
    raise ValueError(
        "Set DATASET_ROOT to the attached dataset directory under /kaggle/input before continuing."
    )
if not DATASET_ROOT.exists():
    raise FileNotFoundError(f'Dataset directory does not exist: {DATASET_ROOT}')

print('Dataset root:', DATASET_ROOT)
print('Extraction root:', EXTRACT_ROOT)

## 2. Find uploaded ZIP files

ZIP discovery is recursive because Kaggle datasets often contain one or more wrapper directories.

In [ ]:
zip_paths = sorted(
    (path for path in ZIP_SEARCH_ROOT.rglob('*') if path.is_file() and path.suffix.lower() == '.zip'),
    key=lambda path: str(path).lower(),
)

if not zip_paths:
    raise FileNotFoundError(f'No ZIP files found recursively under {ZIP_SEARCH_ROOT}')

print(f'Found {len(zip_paths)} ZIP files.')
display(pd.DataFrame({'zip_path': [str(path) for path in zip_paths]}).head(20))

## 3. Parse clip identity from each ZIP name/path

The parser is case-insensitive and tolerates separators such as spaces, `_`, and `-`. It searches the full relative path, so the identity may live in either the ZIP filename or its parent folders. Unmatched ZIPs are shown explicitly for diagnosis.

In [ ]:
ACTIVITY_NAMES = {
    1: 'Falling forward using hands',
    2: 'Falling forward using knees',
    3: 'Falling backwards',
    4: 'Falling sideward',
    5: 'Falling sitting in empty chair',
    6: 'Walking',
    7: 'Standing',
    8: 'Sitting',
    9: 'Picking up an object',
    10: 'Jumping',
    11: 'Laying',
}

FIELD_PATTERNS = {
    'subject': re.compile(r'(?:subject|subj|sub|s)[\s_-]*0*(\d+)', re.IGNORECASE),
    'activity': re.compile(r'(?:activity|act|a)[\s_-]*0*(\d+)', re.IGNORECASE),
    'trial': re.compile(r'(?:trial|try|t)[\s_-]*0*(\d+)', re.IGNORECASE),
    'camera': re.compile(r'(?:camera|cam|c)[\s_-]*0*(\d+)', re.IGNORECASE),
}

def parse_clip_identity(zip_path: Path, root: Path) -> dict:
    relative_text = str(zip_path.relative_to(root).with_suffix(''))
    parsed = {'zip_path': str(zip_path), 'source_name': zip_path.stem}
    for field, pattern in FIELD_PATTERNS.items():
        match = pattern.search(relative_text)
        parsed[field] = int(match.group(1)) if match else None
    return parsed

zip_df = pd.DataFrame(parse_clip_identity(path, ZIP_SEARCH_ROOT) for path in zip_paths)
required_fields = ['subject', 'activity', 'trial', 'camera']
matched_mask = zip_df[required_fields].notna().all(axis=1)

print(f'Parsed completely: {matched_mask.sum()} / {len(zip_df)} ZIPs')
if (~matched_mask).any():
    print('Unmatched ZIPs (inspect their naming pattern before extraction):')
    display(zip_df.loc[~matched_mask])

parsed_zip_df = zip_df.loc[matched_mask].copy()
for field in required_fields:
    parsed_zip_df[field] = parsed_zip_df[field].astype(int)
display(parsed_zip_df.head(20))

## 4. Select the development subset and extract it

Each archive gets a unique output directory based on its source-relative path. Existing complete extractions are reused; a marker file is written only after successful extraction.

In [ ]:
def keep_values(series: pd.Series, allowed):
    return pd.Series(True, index=series.index) if allowed is None else series.isin(allowed)

subset_mask = (
    keep_values(parsed_zip_df['subject'], KEEP_SUBJECTS)
    & keep_values(parsed_zip_df['activity'], KEEP_ACTIVITIES)
    & keep_values(parsed_zip_df['trial'], KEEP_TRIALS)
    & keep_values(parsed_zip_df['camera'], KEEP_CAMERAS)
)
selected_zip_df = parsed_zip_df.loc[subset_mask].reset_index(drop=True)

if selected_zip_df.empty:
    raise ValueError('No ZIPs match the configured development subset.')

print(f'Selected {len(selected_zip_df)} ZIPs for extraction.')
display(selected_zip_df.groupby(['subject', 'activity', 'trial', 'camera']).size().rename('zip_count').reset_index())

In [ ]:
def safe_extract(zip_path: Path, destination: Path) -> None:
    # Reject archive members that would escape the intended destination.
    destination_resolved = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            member_target = (destination / member.filename).resolve()
            if destination_resolved not in member_target.parents and member_target != destination_resolved:
                raise ValueError(f'Unsafe archive member in {zip_path}: {member.filename}')
        archive.extractall(destination)

def extraction_directory(zip_path: Path) -> Path:
    relative = zip_path.relative_to(ZIP_SEARCH_ROOT).with_suffix('')
    return EXTRACT_ROOT / relative

EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
extract_dirs = []

for row in selected_zip_df.itertuples(index=False):
    zip_path = Path(row.zip_path)
    destination = extraction_directory(zip_path)
    complete_marker = destination / '.extraction_complete'

    if not complete_marker.exists():
        if destination.exists():
            shutil.rmtree(destination)
        destination.mkdir(parents=True, exist_ok=True)
        safe_extract(zip_path, destination)
        complete_marker.touch()

    extract_dirs.append(str(destination))

selected_zip_df['extract_dir'] = extract_dirs
print(f'Extraction ready for {len(selected_zip_df)} ZIPs under {EXTRACT_ROOT}.')

## 5. Build frame-level metadata and chronological order

PNG names are sorted with a natural numeric key. This keeps timestamp components numeric—for example, `..._9.png` comes before `..._10.png`. The dataframe contains one row per frame and a zero-based chronological `frame_index` within each clip.

In [ ]:
def timestamp_sort_key(path: Path):
    # Alternating text and integer chunks provide robust chronological sorting
    # for timestamp-style filenames without converting away leading zeros.
    return tuple(
        int(chunk) if chunk.isdigit() else chunk.lower()
        for chunk in re.split(r'(\d+)', path.stem)
    )

frame_rows = []
empty_archives = []

for clip_id, row in selected_zip_df.iterrows():
    extract_dir = Path(row['extract_dir'])
    png_paths = sorted(
        (path for path in extract_dir.rglob('*') if path.is_file() and path.suffix.lower() == '.png'),
        key=timestamp_sort_key,
    )
    if not png_paths:
        empty_archives.append(row['zip_path'])
        continue

    for frame_index, frame_path in enumerate(png_paths):
        frame_rows.append({
            'clip_id': clip_id,
            'subject': row['subject'],
            'activity': row['activity'],
            'trial': row['trial'],
            'camera': row['camera'],
            'activity_name': ACTIVITY_NAMES.get(row['activity'], 'Unknown'),
            'is_fall': int(row['activity'] in {1, 2, 3, 4, 5}),
            'frame_index': frame_index,
            'frame_filename': frame_path.name,
            'frame_path': str(frame_path),
            'zip_path': row['zip_path'],
        })

if empty_archives:
    print(f'Warning: {len(empty_archives)} selected ZIPs contained no PNG files.')
    display(pd.DataFrame({'zip_path': empty_archives}))
if not frame_rows:
    raise RuntimeError('No PNG frames were found in any selected archive.')

metadata_df = pd.DataFrame(frame_rows).sort_values(['clip_id', 'frame_index']).reset_index(drop=True)
metadata_df.to_csv(METADATA_CSV, index=False)

print(f'Built metadata for {len(metadata_df):,} frames across {metadata_df.clip_id.nunique()} clips.')
print('Saved:', METADATA_CSV)
display(metadata_df.head(10))

In [ ]:
clip_summary_df = (
    metadata_df.groupby(
        ['clip_id', 'subject', 'activity', 'trial', 'camera', 'activity_name', 'is_fall'],
        as_index=False,
    )
    .agg(
        num_frames=('frame_path', 'size'),
        first_frame=('frame_filename', 'first'),
        last_frame=('frame_filename', 'last'),
    )
)

display(clip_summary_df)
display(
    clip_summary_df.pivot_table(
        index='subject', columns='activity', values='num_frames', aggfunc='sum', fill_value=0
    )
)

## 6. Visually inspect a sequence

Choose any available subject/activity below. The helper samples evenly across the full clip, making it easy to check whether the ordering follows the motion from beginning to end.

In [ ]:
def show_clip_frames(
    metadata: pd.DataFrame,
    subject: int,
    activity: int,
    trial: int = 1,
    camera: int = 1,
    num_frames: int = 8,
) -> None:
    clip = metadata.loc[
        (metadata['subject'] == subject)
        & (metadata['activity'] == activity)
        & (metadata['trial'] == trial)
        & (metadata['camera'] == camera)
    ].sort_values('frame_index')

    if clip.empty:
        raise ValueError(
            f'No clip found for subject={subject}, activity={activity}, '
            f'trial={trial}, camera={camera}'
        )

    sample_count = min(num_frames, len(clip))
    sample_positions = [round(i * (len(clip) - 1) / max(sample_count - 1, 1)) for i in range(sample_count)]
    sampled = clip.iloc[sample_positions]

    fig, axes = plt.subplots(1, sample_count, figsize=(3.2 * sample_count, 4))
    if sample_count == 1:
        axes = [axes]

    for axis, frame in zip(axes, sampled.itertuples(index=False)):
        with Image.open(frame.frame_path) as image:
            axis.imshow(image.convert('RGB'))
        axis.set_title(f'index {frame.frame_index}\n{frame.frame_filename}', fontsize=8)
        axis.axis('off')

    title = (
        f'S{subject} | A{activity}: {ACTIVITY_NAMES[activity]} | '
        f'T{trial} | C{camera} | {len(clip)} frames'
    )
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

# Start with one fall clip. Change these values after checking availability above.
show_clip_frames(metadata_df, subject=1, activity=1, trial=1, camera=1, num_frames=8)

## Checkpoint before RTMPose

Do not continue until ZIP parsing and chronological RGB visualization look correct. The next cells install RTMPose, process one chosen fall clip, cache `T × 17 × 3` arrays, and draw COCO-17 skeletons. Enable a **GPU accelerator** in Kaggle before continuing.

## 7. Install RTMPose dependencies

Kaggle internet must be enabled for the first run because packages and pretrained weights are downloaded. If imports still fail immediately after installation, use **Run → Restart session**, then rerun the notebook from the imports cell; extracted data under `/kaggle/working` remains available during that session.

In [ ]:
%pip install -q -U openmim
!mim install -q "mmengine>=0.9.0" "mmcv>=2.0.0,<2.2.0"
%pip install -q "mmdet>=3.2.0,<3.4.0" "mmpose>=1.3.0,<1.4.0"

In [ ]:
import json
import pickle

import cv2
import numpy as np
import torch
from mmpose.apis import MMPoseInferencer

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
POSE_CACHE_ROOT = Path('/kaggle/working/up_fall_pose_cache')
POSE_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

print('PyTorch:', torch.__version__)
print('Device:', DEVICE)
if DEVICE == 'cpu':
    print('WARNING: RTMPose will be slow. In Kaggle, select Settings > Accelerator > GPU.')

## 8. Load RTMPose and test one RGB frame

The official `human` alias loads an RTMPose body model and a person detector. This top-down detector-plus-pose setup is preferable to treating the entire image as one person crop. The first execution downloads model weights.

In [ ]:
rtmpose = MMPoseInferencer(pose2d='human', device=DEVICE)

TEST_SUBJECT = 1
TEST_ACTIVITY = 1
TEST_TRIAL = 1
TEST_CAMERA = 1

test_clip_df = metadata_df.loc[
    (metadata_df.subject == TEST_SUBJECT)
    & (metadata_df.activity == TEST_ACTIVITY)
    & (metadata_df.trial == TEST_TRIAL)
    & (metadata_df.camera == TEST_CAMERA)
].sort_values('frame_index').reset_index(drop=True)

if test_clip_df.empty:
    raise ValueError('The configured RTMPose test clip is not present in metadata_df.')

middle_frame = test_clip_df.iloc[len(test_clip_df) // 2]
test_result = next(rtmpose(middle_frame.frame_path, return_vis=True, draw_bbox=True))
print('Detected people:', len(test_result['predictions'][0]))

visualization = test_result['visualization'][0]
plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(visualization, cv2.COLOR_BGR2RGB))
plt.title(f'RTMPose test — frame {middle_frame.frame_index}')
plt.axis('off')
plt.show()

## 9. Extract and cache one pose sequence

UP-Fall normally contains one primary subject. When multiple people are detected, the code initializes from the highest-scoring detection and then selects the candidate whose confident joints are closest to the preceding pose. Missing detections are stored as zeros—never silently copied from another frame.

In [ ]:
def prediction_arrays(prediction: dict):
    keypoints = np.asarray(prediction['keypoints'], dtype=np.float32)
    scores = np.asarray(prediction['keypoint_scores'], dtype=np.float32).reshape(-1)
    if keypoints.ndim == 3 and keypoints.shape[0] == 1:
        keypoints = keypoints[0]
    keypoints = keypoints[:, :2]
    return keypoints, scores

def detection_score(prediction: dict) -> float:
    raw_score = prediction.get('bbox_score', prediction.get('score', 0.0))
    values = np.asarray(raw_score, dtype=np.float32).reshape(-1)
    return float(values[0]) if len(values) else 0.0

def choose_primary_person(predictions, previous_xy=None, previous_scores=None):
    valid = []
    for prediction in predictions:
        xy, scores = prediction_arrays(prediction)
        if xy.shape == (17, 2) and scores.shape == (17,):
            valid.append((prediction, xy, scores))
    if not valid:
        return None
    if previous_xy is None:
        return max(valid, key=lambda item: detection_score(item[0]))

    distances = []
    for item in valid:
        _, xy, scores = item
        visible = (scores >= 0.3) & (previous_scores >= 0.3)
        distance = np.linalg.norm(xy[visible] - previous_xy[visible], axis=1).mean() if visible.any() else np.inf
        distances.append(distance)
    return valid[int(np.argmin(distances))] if np.isfinite(distances).any() else max(valid, key=lambda item: detection_score(item[0]))

def pose_cache_path(subject, activity, trial, camera):
    return POSE_CACHE_ROOT / f'S{subject:02d}_A{activity:02d}_T{trial:02d}_C{camera:02d}.npz'

def extract_pose_sequence(clip_df: pd.DataFrame, overwrite=False) -> Path:
    clip_df = clip_df.sort_values('frame_index').reset_index(drop=True)
    first = clip_df.iloc[0]
    cache_path = pose_cache_path(first.subject, first.activity, first.trial, first.camera)
    if cache_path.exists() and not overwrite:
        print('Using existing cache:', cache_path)
        return cache_path

    keypoints = np.zeros((len(clip_df), 17, 2), dtype=np.float32)
    scores = np.zeros((len(clip_df), 17), dtype=np.float32)
    detected = np.zeros(len(clip_df), dtype=bool)
    previous_xy = previous_scores = None

    for frame_number, frame in enumerate(clip_df.itertuples(index=False)):
        result = next(rtmpose(frame.frame_path, return_vis=False))
        selected = choose_primary_person(result['predictions'][0], previous_xy, previous_scores)
        if selected is not None:
            _, xy, confidence = selected
            keypoints[frame_number] = xy
            scores[frame_number] = confidence
            detected[frame_number] = True
            previous_xy, previous_scores = xy, confidence
        if (frame_number + 1) % 50 == 0 or frame_number + 1 == len(clip_df):
            print(f'Processed {frame_number + 1}/{len(clip_df)} frames', end='\r')

    with Image.open(clip_df.iloc[0].frame_path) as image:
        width, height = image.size
    np.savez_compressed(
        cache_path, keypoints=keypoints, scores=scores, detected=detected,
        frame_paths=clip_df.frame_path.astype(str).to_numpy(),
        frame_indices=clip_df.frame_index.to_numpy(), image_size=np.array([width, height]),
        subject=int(first.subject), activity=int(first.activity), trial=int(first.trial), camera=int(first.camera),
    )
    print(f'\nSaved {cache_path}; detected {detected.mean():.1%} of frames.')
    return cache_path

test_cache_path = extract_pose_sequence(test_clip_df)
test_pose = np.load(test_cache_path)
pose_t17x3 = np.concatenate([test_pose['keypoints'], test_pose['scores'][..., None]], axis=-1)
print('Pose sequence shape:', pose_t17x3.shape)
assert pose_t17x3.shape == (len(test_clip_df), 17, 3)

## 10. Draw cached COCO-17 skeletons

Inspect the whole action: upright stance, descent, contact, and lying. Low-confidence joints are hidden. Red titles indicate frames where no valid person was selected.

In [ ]:
COCO17_EDGES = [
    (0, 1), (0, 2), (1, 3), (2, 4),
    (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),
    (5, 11), (6, 12), (11, 12),
    (11, 13), (13, 15), (12, 14), (14, 16),
]

def draw_pose(axis, image, xy, scores, threshold=0.3):
    axis.imshow(image)
    for start, end in COCO17_EDGES:
        if scores[start] >= threshold and scores[end] >= threshold:
            axis.plot([xy[start, 0], xy[end, 0]], [xy[start, 1], xy[end, 1]], color='lime', linewidth=2)
    visible = scores >= threshold
    axis.scatter(xy[visible, 0], xy[visible, 1], c='red', s=18, edgecolors='white', linewidths=0.5)
    axis.axis('off')

def show_cached_pose_sequence(cache_path: Path, num_frames=10, threshold=0.3):
    pose = np.load(cache_path)
    count = len(pose['keypoints'])
    sample_count = min(num_frames, count)
    positions = [round(i * (count - 1) / max(sample_count - 1, 1)) for i in range(sample_count)]
    fig, axes = plt.subplots(2, (sample_count + 1) // 2, figsize=(18, 8))
    axes = np.asarray(axes).reshape(-1)
    for axis, position in zip(axes, positions):
        with Image.open(str(pose['frame_paths'][position])) as image:
            draw_pose(axis, image.convert('RGB'), pose['keypoints'][position], pose['scores'][position], threshold)
        axis.set_title(f'frame {int(pose["frame_indices"][position])} | mean conf {pose["scores"][position].mean():.2f}',
                       color='black' if pose['detected'][position] else 'red', fontsize=9)
    for axis in axes[len(positions):]:
        axis.axis('off')
    plt.tight_layout()
    plt.show()

show_cached_pose_sequence(test_cache_path, num_frames=10, threshold=0.3)

## 11. Cache all selected clips (run only after visual verification)

Set `RUN_FULL_POSE_EXTRACTION = True` only after the one-clip overlay is correct. This includes hard negatives such as sitting (8) and laying (11). Existing `.npz` caches are reused, so an interrupted run can resume without repeating completed clips.

In [ ]:
RUN_FULL_POSE_EXTRACTION = False

all_cache_paths = []
if RUN_FULL_POSE_EXTRACTION:
    grouped = metadata_df.groupby(['subject', 'activity', 'trial', 'camera'], sort=True)
    for clip_number, (identity, clip_df) in enumerate(grouped, start=1):
        print(f'Clip {clip_number}/{metadata_df.clip_id.nunique()}: {identity}')
        all_cache_paths.append(extract_pose_sequence(clip_df))
    print(f'Cached {len(all_cache_paths)} clips.')
else:
    print('Full extraction is disabled. Verify the test sequence, then set RUN_FULL_POSE_EXTRACTION = True.')

## 12. Build PoseC3D-compatible annotation pickle

Run this after full pose extraction. Labels are zero-based (`activity - 1`). Keypoints have shape `1 × T × 17 × 2`, and scores have shape `1 × T × 17`, matching the MMAction2 pose annotation convention for one tracked person. This only prepares annotations; PoseC3D installation and fine-tuning belong to the next verified notebook stage.

In [ ]:
POSEC3D_ANNOTATION_PATH = Path('/kaggle/working/up_fall_posec3d_annotations.pkl')

cache_files = sorted(POSE_CACHE_ROOT.glob('S*_A*_T*_C*.npz'))
annotations = []
for cache_file in cache_files:
    pose = np.load(cache_file)
    width, height = map(int, pose['image_size'])
    activity = int(pose['activity'])
    annotations.append({
        'frame_dir': cache_file.stem,
        'label': activity - 1,
        'img_shape': (height, width),
        'original_shape': (height, width),
        'total_frames': len(pose['keypoints']),
        'keypoint': pose['keypoints'][None, ...].astype(np.float16),
        'keypoint_score': pose['scores'][None, ...].astype(np.float16),
        'subject': int(pose['subject']),
        'activity': activity,
        'trial': int(pose['trial']),
        'camera': int(pose['camera']),
    })

if annotations:
    with open(POSEC3D_ANNOTATION_PATH, 'wb') as file:
        pickle.dump({'split': {}, 'annotations': annotations}, file)
    print(f'Saved {len(annotations)} annotations to {POSEC3D_ANNOTATION_PATH}')
else:
    print('No pose caches found. Run the one-clip or full extraction cell first.')

## Final verification boundary

Before PoseC3D fine-tuning, inspect at least activities **1, 3, 5, 8, and 11**. Confirm tracking through standing, descent, floor contact, lying, sitting, and already-lying poses. Check missing-frame rates in every cache. Do not interpret results from this three-subject development subset as final evaluation; the serious split must remain subject-independent.